In [1]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class SearchState(TypedDict):
    messages: Annotated[list, add_messages]
    user_query: str      # User requirement summary after LLM understanding
    search_query: str    # Optimized search query for Tavily API
    search_results: str  # Results returned by Tavily search
    final_answer: str    # Final generated answer
    step: str            # Mark current step

#Every important intermediate result is saved.

In [4]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage
from tavily import TavilyClient

load_dotenv()

llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_ID","gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL","https://api.openai.com/v1"),
    temperature=0.7
)


tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

In [ ]:
def understanding_query_node(state: SearchState):
    user_message = state["messages"][-1].content
    # print(state["messages"])
    # print(state["messages"][-1])
    # print(state["messages"][-1].content)
    understand_prompt = f"""Analyze the user's query: "{user_message}"
    Please complete two tasks:
    1. Concisely summarize what the user wants to know
    2. Generate keywords most suitable for search engines (Chinese or English, must be precise)

    Format:
    Understanding: [User requirement summary]
    Search terms: [Best search keywords]"""

    response = llm.invoke([SystemMessage(content=understand_prompt)])

    response_text = response.content
    # print(response)
    # print(response.content)
    search_query = user_message

    if "Search terms:" in response_text:
        search_query = response_text.split("Search terms:")[1].strip()

    return {
        "user_query" : response_text,
        "search_query": search_query,   
        "step": "understood",
        "messages": [AIMessage(content=f"I will search for you: {search_query}")]
    }

In [15]:
test_state = {
    "messages": [
        HumanMessage(content="I only have eggs, tomatoes, and a little cheese in my fridge—what kind of simple dinner can I make?")
    ]
}

understanding_query_node(test_state)

{'user_query': 'Understanding: The user wants a simple dinner recipe using only eggs, tomatoes, and a small amount of cheese.\n\nSearch terms: egg tomato cheese dinner recipe, simple omelette with tomato and cheese, 番茄芝士鸡蛋做法, 鸡蛋番茄奶酪简单晚餐',
 'search_query': 'egg tomato cheese dinner recipe, simple omelette with tomato and cheese, 番茄芝士鸡蛋做法, 鸡蛋番茄奶酪简单晚餐',
 'step': 'understood'}

In [22]:
def tavily_search_node(state: SearchState) -> dict:
    search_query = state['search_query']
    try:
        print(f"🔍 searching {search_query}")
        response = tavily_client.search(
            query=search_query, include_answer=True
        )
        search_results = response["answer"]
        #print(search_results)
        return {
            "search_results" : search_results,
            "step": "searched"
        }


    except Exception as e:
        return {
            "search_results": f"Search failed: {e}",
            "step": "search_failed",
            "messages": [
        AIMessage(content="❌ Search encountered a problem...")
    ]
        }

In [21]:
test_state = {
    "search_query": "easy dinner with eggs tomatoes and cheese"
}

result = tavily_search_node(test_state)

print(result)

🔍 searching easy dinner with eggs tomatoes and cheese
{'search_results': 'A simple dinner with eggs, tomatoes, and cheese can be made by scrambling eggs with melted cheese and cooked tomatoes. Shakshuka is a popular option with poached eggs in a tomato sauce. Another choice is a cheesy tomato egg breakfast.', 'step': 'searched'}
